# Finetuning different types of Transformer Models.

## 01. Causal Language Modeling (CLM)
Used by <b>decoder</b> models like GPT, this approach predicts the next token based on all previous tokens in the sequence. The model can only use context from the left (previous tokens) to predict the next token.

#### Load ELI5 Dataset
Start by loading the first 5000 examples from the <u>ELI5-Category</u> dataset with the 🤗 Datasets library. This’ll give you a chance to experiment and make sure everything works before spending more time training on the full dataset.

In [1]:
from datasets import load_dataset

eli5 = load_dataset("eli5_category", split="train[:5000]", trust_remote_code=True)

# Split the dataset
eli5 = eli5.train_test_split(test_size=0.2)

# Checking one example
eli5["train"][0]

/home/koh/hf-learn-nbs/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'q_id': '5p1t6a',
 'title': 'how did speakers address/were heard when giving speeches to large crowds (like that which will be seen tomorrow) before the inventions of the microphone and speakers systems?',
 'selftext': '',
 'category': 'Physics',
 'subreddit': 'explainlikeimfive',
 'answers': {'a_id': ['dcnrjky', 'dcns8ad', 'dcns1zd'],
  'text': ['The spoke very loudly or used a hand-held megaphone. Also, there were things like cars, planes, and cameras to make random noises... also, people would be quiet.',
   'The trick is to project rather than yell. When you project, you push the sound up from your diaphram, so you can maintain a loud volume for an extended period of time. It involves a lot of abdominal control and a lot of practice to actually pull off successfully, but if you do, a human can fill a BIG space with sound.',
   'Some had naturally loud voices, others trained to project meaning to speak loudly but clearly. This is still a thing for stage actors. Ben Franklin calcula

#### Preprocess

In [2]:
# Load DistilGPT2
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert/distilgpt2"
)

You’ll notice from the example above, the <u>text</u> field is actually nested inside <u>answers</u>. This means you’ll need to extract the <u>text</u> subfield from its nested structure with the [flatten](https://huggingface.co/docs/datasets/process#flatten) method:

In [3]:
eli5 = eli5.flatten()

eli5["train"][0]

{'q_id': '5p1t6a',
 'title': 'how did speakers address/were heard when giving speeches to large crowds (like that which will be seen tomorrow) before the inventions of the microphone and speakers systems?',
 'selftext': '',
 'category': 'Physics',
 'subreddit': 'explainlikeimfive',
 'answers.a_id': ['dcnrjky', 'dcns8ad', 'dcns1zd'],
 'answers.text': ['The spoke very loudly or used a hand-held megaphone. Also, there were things like cars, planes, and cameras to make random noises... also, people would be quiet.',
  'The trick is to project rather than yell. When you project, you push the sound up from your diaphram, so you can maintain a loud volume for an extended period of time. It involves a lot of abdominal control and a lot of practice to actually pull off successfully, but if you do, a human can fill a BIG space with sound.',
  'Some had naturally loud voices, others trained to project meaning to speak loudly but clearly. This is still a thing for stage actors. Ben Franklin calcul

Each subfield is now a separate column as indicated by the <u>answers</u> prefix, and the <u>text</u> field is a list now. Instead of tokenizing each sentence separately, convert the list to a string so you can jointly tokenize them.

Here is a first preprocessing function to join the list of strings for each example and tokenize the result:

In [4]:
def preprocess_function(examples):
    return tokenizer([" ".join(x) for x in examples["answers.text"]])

To apply this preprocessing function over the entire dataset, use the 🤗 Datasets [map](https://huggingface.co/docs/datasets/v4.0.0/en/package_reference/main_classes#datasets.Dataset.map) method. You can speed up the <u>map</u> function by setting <u>batched=True</u> to process multiple elements of the dataset at once, and increasing the number of processes with <u>num_proc</u>. Remove any columns you don’t need:

In [5]:
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=eli5["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]Token indices sequence length is longer than the specified maximum sequence length for this model (1032 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (4145 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1924 > 1024). Running this sequence through the model will result in indexing errors
Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]Token indices sequence length is longer than the specified maximum sequence length for this model (1253 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1213 > 1024). Running this sequence

This dataset contains the token sequences, but some of these are longer than the maximum input length for the model.

You can now use a second preprocessing function to
- concatenate all the sequences
- split the concatenated sequences into shorter chunks defined by <u>block_size</u>, which should be both shorter than the maximum input length and short enough for your GPU RAM.

In [6]:
block_size = 128

def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

# Apply the group_texts function over the entire dataset:
lm_dataset = tokenized_eli5.map(group_texts, batched=True, num_proc=4)

Map (num_proc=4): 100%|██████████| 1000/1000 [00:00<00:00, 4673.38 examples/s]


Now create a batch of examples using [DataCollatorForLanguageModeling](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/data_collator#transformers.DataCollatorForLanguageModeling). It’s more efficient to dynamically pad the sentences to the longest length in a batch during collation, instead of padding the whole dataset to the maximum length.

In [7]:
# Use the end-of-sequence token as the padding token and set mlm=False.
# This will use the inputs as labels shifted to the right by one element:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

#### Train

You’re ready to start training your model now! Load DistilGPT2 with [AutoModelForCausalLM](https://huggingface.co/docs/transformers/v4.53.3/en/model_doc/auto#transformers.AutoModelForCausalLM):

In [8]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2")

At this point, only three steps remain:

1. Define your training hyperparameters in [TrainingArguments](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/trainer#transformers.TrainingArguments). The only required parameter is output_dir which specifies where to save your model. You’ll push this model to the Hub by setting push_to_hub=True (you need to be signed in to Hugging Face to upload your model).

2. Pass the training arguments to [Trainer](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/trainer#transformers.Trainer) along with the model, datasets, and data collator.

3. Call [train()](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/trainer#transformers.Trainer.train) to finetune your model.

In [ ]:
training_args = TrainingArguments(
    output_dir="koh_eli5_clm-model",
    eval_strategy="epoch",
    num_train_epochs=10,
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer.train()

/tmp/ipykernel_35823/4171459427.py:9: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.919500,3.825959
2,3.829900,3.816725
3,3.789800,3.815301


  2025-08-18T23:20:48.749199Z  WARN  Reqwest(reqwest::Error { kind: Request, url: "https://cas-server.xethub.hf.co/xorb/default/79a3e6072dd4a1d93637ebe0b96e0ace53417c97d36b6e9d45167ab94ea827ba", source: hyper_util::client::legacy::Error(SendRequest, hyper::Error(Io, Os { code: 104, kind: ConnectionReset, message: "Connection reset by peer" })) }). Retrying...
    at /home/runner/work/xet-core/xet-core/cas_client/src/http_client.rs:226

  2025-08-18T23:20:48.785377Z  WARN  Reqwest(reqwest::Error { kind: Request, url: "https://cas-server.xethub.hf.co/xorb/default/7733dcfc762fd8e2452bf3dc353b0b4cf3af695065b34c1a2a8116b0be01ae08", source: hyper_util::client::legacy::Error(SendRequest, hyper::Error(Io, Os { code: 104, kind: ConnectionReset, message: "Connection reset by peer" })) }). Retrying...
    at /home/runner/work/xet-core/xet-core/cas_client/src/http_client.rs:226

  2025-08-18T23:20:49.377470Z  WARN  Reqwest(reqwest::Error { kind: Request, url: "https://cas-server.xethub.hf.co/xorb/

Once training is completed, use the [evaluate()](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/trainer#transformers.Trainer.evaluate) method to evaluate your model and get its perplexity:

In [ ]:
import math

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Then share your model to the Hub with the [push_to_hub()](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/trainer#transformers.Trainer.push_to_hub) method so everyone can use your model:

In [ ]:
trainer.push_to_hub()

#### Inference

The simplest way to try out your finetuned model for inference is to use it in a [pipeline()](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/pipelines#transformers.pipeline). Instantiate a pipeline for text generation with your model, and pass your text to it:

In [ ]:
# Prompt you’d like to generate text from...
prompt = "Somatic hypermutation allows the immune system to"

from transformers import pipeline

generator = pipeline("text-generation", model="koh43/koh_eli5_clm-model")
generator(prompt)

In [ ]:
from transformers import AutoTokenizer

# Tokenize the text and return the input_ids as PyTorch tensors:
tokenizer = AutoTokenizer.from_pretrained("koh43/koh_eli5_clm-model")
inputs = tokenizer(prompt, return_tensors="pt").input_ids

Use the [generate()](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/text_generation#transformers.GenerationMixin.generate) method to generate text. For more details about the different text generation strategies and parameters for controlling generation, check out the [Text generation strategies](https://huggingface.co/docs/transformers/generation_strategies) page.

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("koh43/koh_eli5_clm-model")
outputs = model.generate(inputs, max_new_tokens=100, do_sample=True, top_k=50, top_p=0.95)

# Decode the generated token ids back into text:
tokenizer.batch_decode(outputs, skip_special_tokens=True)